In [3]:
import os
import fitz


def document_loader(folder_path):

    document = []

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith('.pdf'):
            file_path = os.path.join(folder_path,file_name)


            pdf = fitz.open(pdf)

            for page_number,page in enumerate(pdf):

                text = page.get_text()

                document.append({
                    'text':text,
                    'page_number':page_number+1,
                    'source':file_name
                })
            pdf.close()

    return document

<h1>Converting each page into Image</H1>

In [4]:
def extract_visual_data(folder_path,output_folder = 'data/images'):

    os.makedirs(output_folder,exist_ok=True)

    images = []

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith('.pdf'):
            file_path = os.path.join(folder_path,file_name)


            pdf = fitz.open(file_path)


            for page_number,page in enumerate(pdf):
                pixmap = page.get_pixmap(matrix = fitz.Matrix(1.5,1.5))

                image_name = (
                    f'{os.path.splitext(file_name)[0]}'
                    f'_page_{page_number+1}'
                )

                image_path = os.path.join(output_folder,image_name)
                
                pixmap.save(image_path)

                images.append({
                    'image_path':image_path,
                    'page number':page_number,
                    'source':file_name
                })



            pdf.close()

        return images


<h1>DECODING THE IMAGES</h1>

In [5]:
from google.genai import types
from google import genai
from dotenv import load_dotenv

client = genai.Client(
    api_key=os.getenv('GEMINI_API_KEY')
)

def analyze_images(image_path):


    with open(image_path,'rb') as file1:
        images_bytes = file1.read()
    prompt = '''
        Analyize this document in detial. 

        Identify:
        1.Images
        2.Graphs
        3.Important text information
        4.Diagrams
        5.Tables
        
        Provide the information represented in a consice but detialed way.

        search for patters, trends in graphs and explain it.
'''

    response = client.models.generate_content(
        model = 'gemini-2.5-flash',
        contents= [
            types.Part.from_bytes(
                data = image_path,
                mime_type='image/jpg'
            ),
            prompt
        ]
    )

    return response.text

<h1>Text Splitter</h1>

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(documents):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap=50
    )

    chunks = []

    for document in documents:
        text = document['text']

        if not text.strip():
            continue

        split_text = splitter.split_text(text)

        for chunk in split_text:
            chunks.append({
                'text':chunk,
                'page':document['page'],
                'source':document['source'],
                'type':'text'
            })
        return chunks

<h1>CREATE EMBEDDING</h1>

In [7]:
from sentence_transformers import SentenceTransformer

class Embedding:
    def __init__(self):
        self.model = 'sentence-transformers/all-MiniLM-L6-v2'

    def embedd(self,text):
        return self.model.encode(
            text,
            convert_to_numpy=True
        )

<h1>CREATE FAISS VECTOR SEARCH</h1>

In [ ]:
import faiss
import numpy as np

class vectorStore:

    def __init__(self,dimension):
        self.index = faiss.IndexFlatL2(dimension)
        self.documents = []

    def add(self,embeddings,document):
        embeddings = np.array(
            embeddings
        ).astype('float32')

        self.index.add(embeddings)
        self.documents.extend(document)
        
    def search(self,query,k=5):
        query_embedded = np.array(
            [query]
        ).astype('float32')

        dist,indices = self.index.search(query_embedded,k)

        result = []

        for index in indices[0]:
            if index<len(self.documents):
                result.append(self.documents[index])
        return result
        

<h1>Build the Multimodal</H1>

In [ ]:
class MultiModelRetriver:
    def __init__(self,documents):
        self.embedding_model = Embedding()

        text = [
            document['text']
            for document in documents
        ]

        embeddingss = self.embedding_model.embed(text)

        dimension = embeddingss.shape[1]

        self.vectr_store = vectorStore(
            dimension
        )

        self.vectr_store.add(embeddingss,documents)

    def retrive(self,query,k=5):
        query_embedding = self.embedding_model.embedd(
            [query]
        )[0]

        return self.vectr_store.search(
            query_embedding,
            k
        )

<h1>MultiModal RAG ENGINE </h1>

In [14]:
class MultiRetriver:

    def __init__(self):
        self.client = genai.Client(
            api_key = os.getenv('GEMINI_API_KEY')
        )

    def generate_answer(self,question,retrived_document):
            context = ""

            for document in retrived_document:

                 context += f'''

source:{document['source']}
page:{document['page']}

content: {document['text']}'''


            prompt = f'''You are a multimodal document analysis assistant

Answer the user's question using ONLY the
provided document context.

If the answer cannot be found in the provided
context, say that the information was not found.

Question:
{question}

Document Context:
{context}

Provide a clear answer.

At the end provide the sources in this format:

Sources:
- document name, page number

'''

    
            response = self.client.models.generate_content(
       model = 'gemini-2.5-flash',
       contents = prompt
    )
            return response.text
